<span style="color:red;font-size:2em;font-weight:bold"> PARTIE 4 - Optimisation des hyperparamètres et du seuil de validation</span>

<span style="color:blue;font-size:1.5em;font-weight:bold;background-color:yellow"> Modules </span>

In [ ]:
# Module pour recharger un module sans redemarrer le kernel
# import importlib
%load_ext autoreload
%autoreload 2

In [ ]:

# Roots
import numpy as np
import pandas as pd
import joblib
import json
import matplotlib.pyplot as plt
from pathlib import Path

from scipy.stats import loguniform

#Selection
from sklearn.model_selection import (
    train_test_split,
    RandomizedSearchCV, 
    StratifiedKFold,
)
# Metrics
from sklearn.metrics import (
    confusion_matrix, fbeta_score,
    precision_score, recall_score,
    make_scorer, ConfusionMatrixDisplay
)

# Feature importance
from sklearn.inspection import permutation_importance

#Modèles
from sklearn.ensemble import (
    HistGradientBoostingClassifier
)

In [ ]:
# Sert à éviter les Warnings avec les transformations sur des vues en transformant 
# ces warning en erreur obligeant ainsi à ne travailler que sur des copies ou les originaux.

pd.set_option('mode.chained_assignment','raise')

In [ ]:
# Ajoute le dossier datas_manipulation au sys.path. Remarque ne pas oublier le __init__.py dans le dossier datas_manipulation
import sys
# root_path = Path(__file__).resolve().parents[1] # Ne fonctionne pas sur notebook
root_path = Path.cwd().parent
sys.path.append(str(root_path))

In [ ]:
# Fonctions personnelles

# utils
from notebooks.utils.features_type_list import features_type
from notebooks.utils.get_feature_names_out import get_feat_names
from notebooks.models_tools.model_attributes import predict_proba_wrapper
from notebooks.models_tools.threshold_tuning import optimize_threshold,cost_function
from notebooks.datas_manipulation.export_datas import export_datas

# pipeline
from notebooks.models_tools.pipeline_builder import build_classification_pipeline

# préprocessing
from notebooks.models_tools.preprocessing import (
    preproc_numerical_features,
    build_preprocessor
)

# modeling
from notebooks.models_tools.modelisation import modeling_cv, predict_models_cv

# plots
from notebooks.plotting.make_model_plots import (
    get_feature_importance, 
    plot_feature_importance, 
    plot_hyperparam_effect,
    pr_curve
)
from notebooks.plotting.config_figures import save_figure


In [ ]:

# Paramètres globaux

# Création dossier results
save_path = root_path.joinpath('datas/results')
Path.mkdir(save_path,exist_ok = True)

# variables globales
random_state=42
cv = StratifiedKFold(
    n_splits=3,
    shuffle=True,
    random_state=random_state
)
# n_jobs: Number of jobs to run in parallel. 
# Training the estimator and computing the score are parallelized 
# over the cross-validation splits. 
# None means 1 unless in a joblib.parallel_backend context. 
# -1 means using all processors.
n_jobs = -1

# test size
test_size = 0.2

# early_stopping validation_fraction
early_stopping_val_frac = 0.1

<span style="color:blue;font-size:1.5em;font-weight:bold;background-color:yellow"> Traquer les expérimentations avec MLFlow </span>

In [ ]:
# import
import mlflow

# ================== Ce qui est dit sur la docs de MLFlow ==============
# 1. Config Tracking
# database en local (sqlite)
# mlflow.set_tracking_uri("sqlite:///mlflow.db")
# mlflow.set_experiment("my-first-experiment")
# database distant
# # Connect to remote MLflow server
# mlflow.set_tracking_uri("http://localhost:5000")
# mlflow.set_experiment("my-first-experiment")
# # ou
# export MLFLOW_TRACKING_URI="http://localhost:5000"
# export MLFLOW_EXPERIMENT_NAME="my-first-experiment"
# =====================================================================

mlflow.set_tracking_uri("sqlite:///mlflow.db")
# On définie l'experiment
experiment_name = "Optimisation_modele"
mlflow.set_experiment(experiment_name)

In [ ]:
# 2. Vérif connexion

print(f"MLflow Tracking URI: {mlflow.get_tracking_uri()}")
print(f"Active Experiment: {mlflow.get_experiment_by_name('Optimisation_modele')}")

# Test logging
with mlflow.start_run():
    mlflow.log_param("test_param", "test_value")
    print("✓ Successfully connected to MLflow!")

**A lancer sur le terminal**
```python
# Accès MLFlow UI

# For Option A (local database)
mlflow server \
    --backend-store-uri sqlite:///mlflow.db \
    --default-artifact-root ./mlruns \
    --host 127.0.0.1 \
    --port 5000
# # For Option B (distant database)
# If you have the remote tracking server running (option C), access the MLflow UI at the same URI.
```

<span style="color:purple"> IMPORTANT - pour moi-même: Suivant la méthodologie employée (utilisation de Kaggle pour la simulation), les résultats de MLFlow sont dans le dossier "**export_mlflow_complet**". Cette précision est nécéssaire car du fait des particlarités de Kaggle (exemple des paths), simplement rappatrier les résultats crééent un conflit au niveau des chemins des fichiers. Pour éviter cela, on a comapcter dans le dossier en question est on utilise la commande:
- **mlflow ui --backend-store-uri sqlite:///mlflow.db**

**On verra ainsi s'afficher les runs, métriques et les sauvegardes autolog. CEPENDANT, ne sera aps présent (présent dans le dossier mais pas UI) les courbes et artefacts**. Cel est dû au fait que mlflow enregistre en dur sur la db (en présence d'une db, il ne crée pas de fichier meta.yaml) d'une façon qui lors de l'export rend difficile la modification afin de changer les chemins de lecture. Ils restent cependant présent en local et en cas de simulation "normal", ils apparaissent bien dans l'UI.


In [ ]:
# Enable autologging for scikit-learn
# Va save le modele, les metriques, les hyperparam et des métadonnées (temps, format...)
mlflow.sklearn.autolog() # type: ignore

<span style="color:blue;font-size:1.5em;font-weight:bold;background-color:yellow"> Datasets </span>

In [ ]:
# Chemin du dataset d'entrainement/test du modèle
datas_path = (
    root_path /'datas'/'raw_datas'/
    'Projet+Mise+en+prod+-+home-credit-default-risk'/'final_datasets'
)

In [ ]:
# Importation de la donnée
# Xy= pd.read_parquet(datas_path/"train.parquet").sample(frac=0.0005, random_state=42) # sample
Xy= pd.read_parquet(datas_path/"train.parquet")
Xy.head()

In [ ]:
Xy.info()

In [ ]:
# Définition des prédicteurs X et de la cible y
X = Xy.drop(columns=['TARGET'])
y = Xy['TARGET']

In [ ]:
# Identification des features numériques et catégorielles
num_list, cat_list = features_type(X)

In [ ]:
# train/test
X_train, X_test, y_train, y_test = train_test_split(
    X, 
    y, 
    test_size=test_size, 
    stratify=y, #Stratifie la proportion de y pour un unique split
    random_state=random_state
)

<span style="color:blue;font-size:1.5em;font-weight:bold;background-color:yellow"> Configurations </span>

<span style="color:blue;font-weight:bold">Modèle</span>

In [ ]:
model = HistGradientBoostingClassifier(
        random_state= random_state,
        class_weight='balanced',
        early_stopping=True,
        validation_fraction=early_stopping_val_frac 
        # sert a valider le early_stopping ou pas en comparant les métriques du jeu-fracton avec la fraction
    )

<span style="color:blue;font-weight:bold">Métriques</span>

In [ ]:
scoring = {
        'f2':make_scorer(fbeta_score, beta=2, zero_division=0),
        'precision':make_scorer(precision_score, zero_division=0),
        'recall':make_scorer(recall_score, zero_division=0),
    }

<span style="color:blue;font-weight:bold">Pipeline et grille de distribution</span>

Remarque: Si on était resté sur la même session/notebook que la partie précédente il n'aurait pas été nécéssaire de redéfinir la pipeline et il suffirait de rappelé la pipeline du candidat choisit (candidate_pipe = pipelines[candidate])

In [ ]:
# Preprocessing

# Matrice dense sans scaling (RF, GradientBoosting, HistGB)
no_scaling = preproc_numerical_features()

preproc_noScale_dense = build_preprocessor(
    numeric_features=num_list, 
    categorical_features=cat_list,
    num_pipeline=no_scaling,
    sparse_output=False
)

In [ ]:
# Pipeline

hgb_pipe = build_classification_pipeline(model,preproc_noScale_dense)

In [ ]:
# Grille de distribution des hyperparamètres d'études
prefix_step = "classifier__" # Pipeline == prefixe complementaire suivant la profondeur
param_distrib = {
    f'{prefix_step}max_depth':np.arange(3,12), # Profondeur de l'arbre /  risque d'overfit si important (defaut None)
    f'{prefix_step}learning_rate':loguniform(0.02,0.08), # Influence vit d'apprent/ Très sensible / bas = stable mais couteux (defaut 0.1)
    f'{prefix_step}max_iter' :np.arange(100,800,100),# Nb max d'arbres / 300---500 par pas de 100 / à combiner avec early_stopping (defaut 100)
    f'{prefix_step}l2_regularization':loguniform(1e-3,10.0), # Coeff pour le terme L2 de regul de la fonction de cout (defaut 0)
    # (ecart entre y_pred et y_test pdt train) 
    # Controle l'ajustement du modèle face aux données notamment du bruit)
    f'{prefix_step}min_samples_leaf':[20,50,100,200,400]
}

Remarque: En présence de pipeline, le nom de l'étape (preproc, sampler, model) est rattaché comme suffix aux hyperparamètres. plus il y a de sous-pipeline plus le prefix grandit.

<span style="color:blue;font-size:1.5em;font-weight:bold;background-color:yellow"> Modélisation </span>

In [ ]:
# # On définie l'experiment
# experiment_name = "Optimisation_modele"
# mlflow.set_experiment(experiment_name)

In [ ]:
# # Désactive le log des modèles pendant la CV pour éviter les sous-runs
# mlflow.sklearn.autolog(disable = True) #type:ignore

In [ ]:
# START_RUN
mlflow.start_run(run_name="HGB_optimisation_hyperparamètres")

<span style="color:blue;font-size:1.5em;font-weight:bold;background-color:yellow">Optimisation des hyperparamètres</span>

Pour étudier les hyperparam afin de sotir le meilleur modèle parmi ceux testé on va devoir réaliser des combinaisons de ces paramètres, modéliser et comparer les performances.

Dans notre cas, on a 5 hyperparamètres, sans compter les loguniform (disons 5 valeurs) on aurait $5x5x7x5x5 = 4375$ combinaisons multiplié par cv = 3 et hypothétiquement 100 secondes en moyennes par simulation ( temps de base du hgb précédemment) il faudrait environ 364 heures de calculs ==> IMPOSSIBLE. Solution? On utilise à la place de GridSearchCV, **RandomizedSearchCV**, on peut ainsi limiter le nombre de combinaisons et l'aléatoire se chargera de choisir les combinaisons dans le spectre donné.

In [ ]:
# ==================== Configuration de RSCV ====================
random_search = RandomizedSearchCV(
    estimator = hgb_pipe,
    param_distributions = param_distrib,
    scoring = scoring,
    refit = 'f2', # choix de la métrique de  controle
    n_iter  = 20, # Contrôle le nombre de combinaison max
    cv = cv,
    n_jobs = n_jobs, # calcul parallèle
    random_state = random_state,
    return_train_score = True,
    verbose = 2, # Affiche temp de calcul pour chaque fold avec param candidat + le score
)

In [ ]:
# Entrainement et determination de l'optimum
random_search.fit(X_train,y_train)

In [ ]:
# ======================= LOGGING ======================================
best_model_optHyperparams = random_search.best_estimator_

# Logging des meilleurs params avec opt des hyperparams
mlflow.log_params(random_search.best_params_)
mlflow.log_metric("optHyperparams_f2", random_search.best_score_)
mlflow.sklearn.log_model(sk_model=best_model_optHyperparams, #type:ignore
    name="best_model",
    registered_model_name = "best_model", # Permet de faire passer le modele dans le registry
)
mlflow.set_tag("optHyperparams", "Résultats avec HGB après optimisation hyperparams")

In [ ]:
# Sauvegarde local
# model
model_path = save_path/"best_model"
model_path.mkdir(parents=True, exist_ok=True)
joblib.dump(best_model_optHyperparams, model_path/'best_model.joblib')

<span style="color:blue;font-weight:bold"> Visualisation </span>

In [ ]:
# ===================== VISUALISATION & METRIQUES ============================
# On construit la df des résultats du RSCV
rscv_results = pd.DataFrame(random_search.cv_results_)

# plot des effets des hyperparam
hyperparam_fig = plot_hyperparam_effect(
    rscv_results,
    model_type="classification",
    title_save="hyperParamEffect",
    save_path = save_path/"figures"
)


plt.show()

In [ ]:
# ======================= LOGGING ======================================

# sauvegarde de la figure dans les artefacts de mlflow
mlflow.log_artifact(save_path/"figures/hyperParamEffect") # type: ignore

Le graphique généré montre l'influence d'un hyperparamètre sur les métriques choisies, chaque ligne représentant une métrique (de haut en bas on a f2, precision puis recall) et chaque colonne, un hyperparamètre (gauche a droite: l2, learning_rate, max_depth, max_iter et min_sample_leaf).
- les courbes n'évoluent pas linéairement
- les loguniform (learning_rate et l2) ont une concentration de valeurs autour du range min et on s'apperçoit d'une fluctuation des performances en dent de scie. 
- Ce sont aussi les seuls à ne pas avoir de fluctuation (bande bleue) entre les tentatives.

In [ ]:
# END_RUN()
mlflow.end_run()

In [ ]:
# # réactive le log des modèles après CV
# mlflow.sklearn.autolog(disable = False) #type:ignore

<span style="color:blue;font-size:1.5em;font-weight:bold;background-color:yellow">Optimisation du seuil de validation</span>

In [ ]:
# # Désactive le log des modèles pendant la CV pour éviter les sous-runs
# mlflow.sklearn.autolog(disable = True) #type:ignore

In [ ]:
# START_RUN
mlflow.start_run(run_name="HGB_optimisation_threshold")

In [ ]:
# ===================== PREDICT PROBA ============================
# calcul des probas sur le jeu holdout/test
y_test_pred_proba = random_search.predict_proba(X_test)[:,1]

# calcul des courbes alignées
precisions, recalls, thresholds, pr_auc = pr_curve(y_test, y_test_pred_proba)

# ====================== sans fonction de cout/ cout métier explicite
# # Optimisation du seuil
# pred_proba_results = optimize_threshold(
#     precisions, 
#     recalls,
#     thresholds,
#     method="fbeta", 
#     beta=2.0
# )
# optimal_thresh = pred_proba_results['optimal_threshold']

# ============ En prenant en compte le coût métier ================
thresholds_list = np.linspace(0.1, 0.9, 801)
costs = []

for threshold in thresholds_list:
    # Calcul du coût pour chaque seuil
    y_pred = (y_test_pred_proba >= threshold).astype(int)
    cost,tn, fp, fn, tp = cost_function(y_test, y_pred,cost_fn=10,cost_fp=1)
    costs.append(cost)
# indice associé à la valeur min de cout
best_idx = int(np.argmin(costs))

# threshold équivalent
optimal_thresh = float(thresholds_list[best_idx])
# coût associé
min_cost = float(costs[best_idx])

# f2 au seuil optimal de coût pour le logging
y_test_pred = (y_test_pred_proba >= optimal_thresh).astype(int)
f2_test = float(fbeta_score(y_test, y_test_pred, beta=2))

In [ ]:
# ======================= LOGGING ======================================

mlflow.log_metric("pr_auc", pr_auc)
mlflow.log_metric("optThreshold", float(optimal_thresh))
# =============
# mlflow.log_metric("f2_associe", float(pred_proba_results["best_f_score"]))
mlflow.log_metric("f2_associe", f2_test)
# =============

In [ ]:
# ================ SAUVEGARDE LOCALE =================
opt_thresh_path = save_path/"opt_thresh"
opt_thresh_path.mkdir(parents=True, exist_ok=True)
config = {
    "threshold": float(optimal_thresh),
    "model": "HistGradientBoosting",
    "pr_auc":pr_auc,
    # =============
    # "f2":float(pred_proba_results["best_f_score"])
    "f2":f2_test
    # =============
}
with open(opt_thresh_path/"opt_thresh.json", "w") as f:
    json.dump(config, f)

<span style="color:blue;font-weight:bold"> Visualisation </span>

In [ ]:
# ===================== VISU & METRIQUES ============================
# Tracé
threshold_fig, ax = plt.subplots(figsize=(12,10))
ax.plot(recalls, precisions, label=f'HGB avec hyperparams opt (PR-AUC = {pr_auc:.3f})')
# =============
# ax.scatter(
#     pred_proba_results["recall"], pred_proba_results["precision"], 
#     c='red', s=100, 
#     label=f"Seuil optimal",
# )
tn_opt, fp_opt, fn_opt, tp_opt = confusion_matrix(y_test, y_test_pred).ravel()
ax.scatter(
    tp_opt/(tp_opt+fn_opt), tp_opt/(tp_opt+fp_opt), # Recall, Precision
    c='red', s=100, 
    label=f"Seuil coût optimal ({optimal_thresh:.3f})",
)
# =============
ax.set_xlabel('Recall')
ax.set_ylabel('Precision')
ax.set_title('PR Curve')
ax.legend()

plt.tight_layout()
plt.show()

In [ ]:
# ======================= LOGGING ======================================

# mlflow.log_artifact(str(save_path/"figures/comparaison_modeles_courbes_pr"))
mlflow.log_figure(threshold_fig, "optThreshold_pr_curve.png")

In [ ]:
# ======================= SAUVEGARDE LOCALE =========================

# Sauvegarde de la courbe
save_figure("optThreshold_pr_curve", save_path/"figures")

La courbe PR (plus fiable que le ROC car déséquilibre très fort de classe d'où le choix du PR plutôt que le ROC comme sur Kaggle) montre une courbe décroissante et presque lineaire si on vers 0.1 de recall (erratique à très faible recall en plus du désalignement sur 1), cela montre justement qu'il a été judicieux d'entrainer le montrer sur le score f2 car bien qu'on sache via la logique métier qu'on veut prioriser le recall, il ne reste pas acceptable d'avoir une precision proche de 0 (FP >> TP le modèle pense que personne n'est solvable). Grâce à cela, le modèle a appris a prioriser le recall et via la fonction de coût, on trouve un seuil optimisé a 0.505 ce qui donne un recall d'environ 0.68 (point rouge).



**Remarque**: on a utilisé une méthode standard pour l'optimisation: apprentissage interne (log-loss intégré), comparaison des métriques et choix (scoring et refit sur f2), coût métier. Dans notre cas en plus, sklearn + HGB sont assez rigide sur certains points (fonction de coût interne no customisable et pondération limitée: sample_weight VS class_weight). Si ça avait été catboost, on aurait pu pondéré réellement les classes et si ça avait été xgboost, on aurait eu encore plus de liberté en ayant la possibilité de créer la custom cost function à injecter pendant l'entrainement (ou plus simplement mais limité en utilisant le scale_pos_weight).

<span style="color:blue;font-weight:bold"> Confirmation d'overfitting </span>

In [ ]:
# ======================== Recherche d'overfitting ===============================
# f2 score rscv, le f2 donné par le meilleur modele avec hyperparamètres optimisés
f2_rscv = random_search.best_score_
# Calcul du Gap
gap = abs(f2_rscv - f2_test)


In [ ]:
# ======================= LOGGING ======================================

mlflow.log_metric("overfitting_gap", float(gap))
    
print(f" Analyse Overfitting (Métrique: F2-Score) :")
print(f"   - F2 RSCV (train) : {f2_rscv:.3f}")
print(f"   - F2 de référence (test) : {f2_test:.3f}")
print(f"   - Écart (Gap)          : {gap:.3f}")
if gap >=0.03:
    print('Overfitting potentiel')
    mlflow.set_tag("Status","Risque d'overfitting")
else:
    mlflow.set_tag("Status", "Pas d'overfitting")

- f2_rscv renvoie la performance théorique du meilleur candidat.
- f2_test exprime la performance réelle attendue "dans la vraie vie".

Un écart (>0.03) va traduire un potentiel overfitting 

On note un écart de 0.08 ==> pas d'overfit

<span style="color:blue;font-weight:bold"> Matrice de confusion</span>

In [ ]:
# ================ MATRICE DE CONFUSION (JEUX TEST/THRESHOLD OPT) ====================

# Prédiction avec seuil optimisé
# y_test_pred = (y_test_pred_proba >= optimal_thresh).astype(int) # deja présent plus haut
# prédiction avec seuil par défaut
y_test_pred_default = (y_test_pred_proba == 0.5).astype(int)

# matrice des confusion avec seuil par défaut et opt
#defaut
confusionMatrix_default = confusion_matrix(y_test, y_test_pred_default)
confusionMatrix_plot_default = ConfusionMatrixDisplay(
    confusion_matrix=confusionMatrix_default, 
    display_labels=['Solvable', 'Risque']
)
# opt
confusionMatrix = confusion_matrix(y_test, y_test_pred)
confusionMatrix_plot = ConfusionMatrixDisplay(
    confusion_matrix=confusionMatrix, 
    display_labels=['Solvable', 'Risque']
)

<span style="color:blue;font-weight:bold"> Visualisation </span>

In [ ]:
# ===================== VISU ==========================

confusionMatrix_fig, (ax_def, ax_opt) = plt.subplots(2, 1, figsize=(6, 12))

# defaut
confusionMatrix_plot_default.plot(ax=ax_def, cmap='Greys', colorbar=False)
ax_def.set_title(f"Seuil 0.5 et coût {fn*10+fp}", fontsize=12)

# opt
confusionMatrix_plot.plot(ax=ax_opt, cmap='Blues', colorbar=False)
ax_opt.set_title(f"Seuil {optimal_thresh} et coût {int(min_cost)}", fontsize=12)

plt.suptitle(f"Matrice de confusion (seuil par défaut (0.5) VS {optimal_thresh:.3f})")
plt.tight_layout()


plt.show()

In [ ]:
# ======================= LOGGING ======================================


mlflow.log_figure(confusionMatrix_fig, "Matrice_confusion_jeux_test.png")

In [ ]:
# ======================= SAUVEGARDE LOCALE =========================

# Sauvegarde de la courbe
save_figure("Matrice_confusion_jeux_test", save_path/"figures")

pour rappel:
- TP: client NON solvable et reconnu par le modèle
- TN: client solvable et detecté par le modèle
- FN: client NON solvable NON détécté par le modèle
- FP: client solvable NON détécté par le modèle

| TN (bon client) | FP (refusé à tort)
|----|------
| **FN (danger)** | **TP (refusé à raison)**

avec un FN >= 10x FP en terme de cout pour la banque.

La matrice du haut représente le cas avec un seuil par défaut (0.5) tandis que celui du bas, la matrice avec seuil optimisé.
- Le seuil n'a que légèrement changé
- Le cas par défaut range tout le monde en solvable (classe 0/ négatif)
- Le cout passe de 48997 a 30348 soit une amélioration de 38% déjà (on réduit 1/3 les coût financiers lié aux erreurs FP-FN!)

In [ ]:
# END_RUN()
mlflow.end_run()

In [ ]:
# # réactive le log des modèles après CV
# mlflow.sklearn.autolog(disable = False) #type:ignore

Avant de passer aux features importances via shap, on a relancer un run avec des ranges d'hyperparamètres plus optimisés:

```python
param_distrib = {
    f'{prefix_step}max_depth':np.arange(6,15), # Profondeur de l'arbre /  risque d'overfit si important (defaut None)
    f'{prefix_step}learning_rate':loguniform(0.01,0.03), # Influence vit d'apprent/ Très sensible / bas = stable mais couteux (defaut 0.1)
    f'{prefix_step}max_iter' :np.arange(500,1000,100),# Nb max d'arbres / 300---500 par pas de 100 / à combiner avec early_stopping (defaut 100)
    f'{prefix_step}l2_regularization':loguniform(1e-3,1.0), # Coeff pour le terme L2 de regul de la fonction de cout (defaut 0)
    # (ecart entre y_pred et y_test pdt train) 
    # Controle l'ajustement du modèle face aux données notamment du bruit)
    f'{prefix_step}min_samples_leaf':randint(30, 150)
}
```

Et on en a tiré les observations et conclusions suivantes:
1. **Concernant les hyprparamètres**

- On a une fluctuation sur l2 et le min_sample_leaf
- l2, learning_rate et min_sample_leaf n'ont pas de range (bande bleue)
- la fluctuation sur l'intervalle des métriques est réduite avec des max similaires ce qui suppose qu'on a, à minima, conserver la qualité au modèle si ce n'est améliorer.

2. **PR_AUC**

- Même tendance et même quasi identique avec une évolution légère de l'AUC (0.270) et du seuil (0.511)
- Réduction de l'écart f2 rscv/base à 0.005 (==> pas d'overfit)
- Cout avec seuil de défaut à 49108 VS optimisé à 30515 soit amélioration de 37.8%.

**On a légèrement améliorer les résultats mais le coût de simulation montre que ça n'en valait pas forcément la peine (1.9h VS 4.2h). On partira tout de même sur ce modèle pour la suite**


